In [1]:
from pathlib import Path


from usas_validator.utils import load_usas_mapper

from usas_evaluation_framework.dataset import EvaluationDataset
from usas_evaluation_framework.parsers.benedict import EnglishBenedict, FinnishBenedict
from usas_evaluation_framework.parsers.torch import TorchParser
from usas_evaluation_framework.parsers.corcencc import CorcenccParser
from usas_evaluation_framework.parsers.icc_irish import ICCIrishParser
from usas_evaluation_framework.parsers.naacl_2015 import NAACL2015USAS
from usas_evaluation_framework.parsers.danish_wikipedia import DanishWikipediaUSAS
from usas_evaluation_framework.parsers.dutch_wikipedia import DutchWikipediaUSAS
from usas_evaluation_framework.parsers.hindi_wikipedia import HindiWikipediaUSAS
from usas_evaluation_framework.parsers.spanish_wikipedia import SpanishWikipediaUSAS
from usas_evaluation_framework.parsers.english_wikipedia import EnglishWikipediaUSAS
from usas_evaluation_framework.parsers.igbo_wikipedia import IgboWikipediaUSAS

In [3]:
dataset_name_parser = [
    ("benedict/english/benedict_english_corpus.txt", "English", EnglishBenedict),
    ("benedict/finnish/benedict_finnish_corpus.txt", "Finnish", FinnishBenedict),
    (["icc_irish/ICC-GA-WPH-001-the_wire.tsv", "icc_irish/ICC-GA-WPH-003-george_orwell.tsv", "icc_irish/ICC-GA-WR0-021-tuairisc.tsv"], "Irish", ICCIrishParser),
    ("corcencc/corcencc_corpus.txt", "Welsh", CorcenccParser),
    ("torch/torch_corpus.csv", "Chinese", TorchParser),
    ("naacl_2015_chinese/naacl_2015_chinese_corpus.csv", "Chinese-N", NAACL2015USAS),
    ("naacl_2015_portuguese/naacl_2015_portuguese_corpus.csv", "Portuguese", NAACL2015USAS),
    ("naacl_2015_italian/naacl_2015_italian_corpus.csv", "Italian", NAACL2015USAS),
    ("spanish_wikipedia/spanish_wikipedia_corpus.csv", "Spanish", SpanishWikipediaUSAS),
    ("hindi_wikipedia/hindi_wikipedia_corpus.csv", "Hindi", HindiWikipediaUSAS),
    ("dutch_wikipedia/dutch_wikipedia_corpus.csv", "Dutch", DutchWikipediaUSAS),
    ("danish_wikipedia/danish_wikipedia_corpus.csv", "Danish", DanishWikipediaUSAS),
    ("english_wikipedia/english_wikipedia_corpus.csv", "English-M", EnglishWikipediaUSAS),
    ("igbo_wikipedia/igbo_wikipedia_corpus.csv", "Igbo", IgboWikipediaUSAS),
]

In [4]:
dataset_directory = Path().cwd().parent / "tests" / "data" / "parsers"
labels_to_filter = {"Z9", "Z99"}
valid_usas_labels = set(load_usas_mapper(None, None).keys())

dataset_df_dict = {
    "Language": [],
    "Text Level": [],
    "Texts": [],
    "Tokens": [],
    "L. Tokens": [],
    "Multi Tag Membership (%)": [],
    "Unique Semantic Tags": [],
    "Multi Word Expressions": []
}

for data_path, dataset_name, parser in dataset_name_parser:
    dataset: None | EvaluationDataset = None
    if isinstance(data_path, list):
        all_datasets = []
        for data_path_item in data_path:
            dataset_path = dataset_directory / data_path_item
            all_datasets.append(parser.parse(dataset_path, dataset_name=None, label_filter=labels_to_filter, label_validation=valid_usas_labels))
        dataset = EvaluationDataset.merge(dataset_name, all_datasets[0].text_level, None, *all_datasets)
    else:
        dataset_path = dataset_directory / data_path
        dataset = parser.parse(dataset_path, dataset_name=dataset_name, label_filter=labels_to_filter, label_validation=valid_usas_labels)
    assert isinstance(dataset, EvaluationDataset)

    dataset_statistics = dataset.stats()
    dataset_df_dict["Language"].append(dataset_name)
    dataset_df_dict["Text Level"].append(dataset.text_level.value)
    dataset_df_dict["Texts"].append(len(dataset))

    number_tokens = 0 if dataset_statistics.num_tokens is None else dataset_statistics.num_tokens
    dataset_df_dict["Tokens"].append(number_tokens)

    number_labelled_tokens = 0 if dataset_statistics.num_labelled_tokens is None else dataset_statistics.num_labelled_tokens
    dataset_df_dict["L. Tokens"].append(number_labelled_tokens)
    
    number_compound_tags = 0 if dataset_statistics.num_compound_semantic_tags is None else dataset_statistics.num_compound_semantic_tags
    num_compound_semantic_tags_percent = "0%"
    if number_compound_tags != 0:
        num_compound_semantic_tags_percent_float = (
            number_compound_tags
            / number_labelled_tokens
            * 100
        )
        num_compound_semantic_tags_percent = f"{num_compound_semantic_tags_percent_float:.1f}%"
    dataset_df_dict["Multi Tag Membership (%)"].append(f"{number_compound_tags}({num_compound_semantic_tags_percent})")
    
    unique_semantic_tags = dataset_statistics.unique_semantic_tags if dataset_statistics.unique_semantic_tags is not None else frozenset()
    dataset_df_dict["Unique Semantic Tags"].append(len(unique_semantic_tags))

    number_mwes = dataset_statistics.num_mwes if dataset_statistics.num_mwes is not None else 0
    dataset_df_dict["Multi Word Expressions"].append(number_mwes)

In [5]:
import pandas

print(pandas.DataFrame(dataset_df_dict).to_latex(index=False))

\begin{tabular}{llrrrlrr}
\toprule
Language & Text Level & Texts & Tokens & L. Tokens & Multi Tag Membership (%) & Unique Semantic Tags & Multi Word Expressions \\
\midrule
English & sentence & 73 & 3899 & 3468 & 212(6.1%) & 149 & 137 \\
Finnish & sentence & 72 & 2439 & 2068 & 254(12.3%) & 144 & 30 \\
Irish & paragraph & 3 & 707 & 617 & 60(9.7%) & 101 & 24 \\
Welsh & sentence & 611 & 14876 & 12803 & 1314(10.3%) & 199 & 0 \\
Chinese & sentence & 46 & 2312 & 1756 & 1(0.1%) & 128 & 0 \\
Chinese-N & sentence & 35 & 1056 & 512 & 56(10.9%) & 115 & 0 \\
Portuguese & sentence & 39 & 1232 & 781 & 26(3.3%) & 119 & 18 \\
Italian & sentence & 215 & 4499 & 1692 & 79(4.7%) & 135 & 18 \\
Spanish & sentence & 30 & 1533 & 1390 & 56(4.0%) & 115 & 45 \\
Hindi & sentence & 80 & 2013 & 1810 & 2(0.1%) & 113 & 267 \\
Dutch & sentence & 58 & 1088 & 944 & 32(3.4%) & 113 & 2 \\
Danish & sentence & 58 & 1104 & 949 & 5(0.5%) & 99 & 32 \\
English-M & sentence & 166 & 4036 & 3506 & 0(0%) & 131 & 582 \\
Igbo & sente